<div align="center">
  <img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" alt="Wayne State University Logo" width="320">
  <h1>Homework 2: Scientific Computing,<br>Manufacturing Signals, and Feature Engineering</h1>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/homework/hw02_scientific_computing_manufacturing_signals_and_feature_engineering.ipynb)

[Course Repository](https://github.com/WSU-AI-in-ME/ai-in-me-1) · [Homework Index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Homework_index.ipynb)

**ME 5995 — AI in Mechanical Engineering I: Fundamentals of Manufacturing Data Science**  
**Wayne State University**

**Version:** Student Version

**Total: 100 points · Approximately 85 minutes (60–90 minute range).**

## Student Information

Edit this Markdown cell.

**Student name:** TODO: Enter your name  
**WSU AccessID:** TODO: Enter your WSU AccessID

## Purpose and learning objectives
Follow one milling record from sampled data to features, then connect a cut-level
feature table to wear. Review Labs 2–3; use your own observations and reasoning.
You will identify sample rows/units, build physical time, compute four scalar
features, interpret FFT/STFT, and justify three features using correlation and
redundancy. No model or data split is required.

Complete **10 short code entries and five short responses**. Supplied code is
run unchanged, but its required outputs and interpretations are assessed.
Parts 1–5 take approximately 10/10/15/20/20 minutes; reserve 10 minutes to check
and submit. This is an estimate, not a timed exercise.

Use NumPy, pandas, SciPy and Matplotlib. Install required packages in your local
Python environment; if requirements.txt is provided, you may use it. In Colab,
try the imports without adding unnecessary installation steps.

This homework uses PHM Society 2010 milling data: c1 cut 315 and the course
945-cut master, with cut 1 supplied only for the Part 4 comparison. See the [PHM data card](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/README.md)
for attribution, provenance and rights, and the [feature dictionary](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/DATA_DICTIONARY.md)
for definitions. Cut numbers are sequence positions, not wear classes.

## Provided setup

Run unchanged. `None` marks an unfinished answer; reminders do not mean the homework is complete.

In [ ]:
# [GUIDED - RUN UNCHANGED]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.fft import rfft, rfftfreq
from scipy.signal import spectrogram
from scipy.signal.windows import hann
from IPython.display import display
# Supplied display settings keep plots readable without changing calculations.
plt.rcParams.update({'font.size':12, 'axes.titlesize':16, 'axes.labelsize':14,
                     'xtick.labelsize':12, 'ytick.labelsize':12, 'legend.fontsize':12,
                     'figure.figsize':(9,4), 'figure.dpi':120})
pd.set_option('display.precision', 4)

## How to work through this homework
1. Run **GUIDED - RUN UNCHANGED** cells in order. These contain finished data and plotting code.
2. In a task cell, replace only `None` after `=` with the requested expression.
   Keep quotation marks around a column name; numerical expressions need no quotes.
3. Run that cell and the next supplied output cell. If a reminder appears, return to the earlier input.
4. Read the response guide, inspect your own outputs, and write within the sentence limit.

This homework transfers the Lab methods to **Vibration Y**, compares it with a
provided X reference, and explores **all seven sensor channels**. You will not
write FFT/STFT, heatmap loops, or 28 separate correlation expressions.

# Part 1 — Load and Inspect Manufacturing Data — 15 points

**Time guide: 10 minutes.**

**Read first:** `data.shape` gives (number of rows, number of columns).
Find the name containing `vibration_y` in the displayed columns. Copy that full
name inside quotes. `len(data)` counts rows. Divide this count by `fs` for nominal
duration in seconds; the last sample occurs one sampling interval earlier.

In [ ]:
# [GUIDED - RUN UNCHANGED]
# DATA_URL locates one whole cut; data stores samples as rows and sensors as columns.
DATA_URL = 'https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/data/phm2010/c1_selected_cuts/c1_cut315.csv'
data = pd.read_csv(DATA_URL, float_precision='round_trip')
fs = 50_000  # Samples per second: 50,000 simultaneous readings each second.
display(data.head(3))
print('Shape (samples, channels):', data.shape)
print('Column names:', data.columns)
# Completed column-name example; identify Vibration Y from the displayed names.
force_column = 'force_x_N'

In [ ]:
# Choose the acceleration channel; N counts sample rows, duration_s is in seconds.
vibration_column = None  # TODO: Enter the Vibration Y column name.
N = None  # TODO: Count sample rows using len(data).
duration_s = None  # TODO: Divide the sample count by fs.

In [ ]:
# [GUIDED - RUN UNCHANGED]
print('Selected column:', vibration_column)
print('Sample count:', N)
print('Nominal duration (s):', duration_s)

## Part 1 response (1–2 sentences)

What does one row of this DataFrame represent?

**TODO:** Write your response here.

# Part 2 — Physical Time and Signal Visualization — 20 points

**Time guide: 10 minutes.**

**Build physical time:** the sample index counts readings; it is not in seconds.
Follow the five-sample example, then apply the same division to `sample_index`.
The complete time array must have one entry for every acceleration sample.
Read the resulting plot left to right; compare the width of the oscillating band
in an early interval and near the end, without assigning a mechanical cause.

In [ ]:
# [GUIDED - RUN UNCHANGED]
# Tiny completed example: sample numbers divided by samples/second give seconds.
example_time = np.arange(5) / fs
print('Five sample times (s):', example_time)
sample_index = None
vibration_y = None
if N is not None and vibration_column is not None:
    sample_index = np.arange(N)  # Numbers 0, 1, ..., N-1 for the selected record.
    vibration_y = data[vibration_column].to_numpy()  # Acceleration samples in g.
else:
    print('Complete the column and sample count in Part 1 first.')

In [ ]:
# time_s must give the elapsed time of every sample in seconds.
time_s = None  # TODO: Convert sample_index to seconds using fs.

In [ ]:
# [GUIDED - RUN UNCHANGED]
if time_s is None or vibration_y is None:
    print('Complete the earlier required inputs, then rerun this cell.')
else:
    # fig is the whole figure; ax is its plotting area. This plot uses the full record.
    fig, ax = plt.subplots()
    ax.plot(time_s, vibration_y, linewidth=0.3)
    ax.set(title='c1 cut 315: Vibration Y', xlabel='Time (s)', ylabel='Vibration Y (g)')
    fig.tight_layout()
    plt.show()

## Part 2 response (1–2 sentences)

Describe one directly visible characteristic of the waveform. Do not infer a physical cause.

**TODO:** Write your response here.

# Part 3 — Time-Domain Features — 25 points

**Time guide: 15 minutes.**

![One Signal, Four Ways to Summarize It](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab03/four_features.png)

Synthetic conceptual example; exact definitions follow.

**Mean:** average signed level. **Sample SD:** fluctuation around the mean.
**RMS:** overall magnitude relative to zero. **Peak-to-peak:** observed full range.
All four retain the sensor unit (g for Vibration Y).

$$\bar{x}=\frac{1}{N}\sum_{i=1}^{N}x_i,\qquad
s=\sqrt{\frac{1}{N-1}\sum_{i=1}^{N}(x_i-\bar{x})^2}$$
$$x_{\mathrm{RMS}}=\sqrt{\frac{1}{N}\sum_{i=1}^{N}x_i^2},\qquad
x_{p-p}=x_{\max}-x_{\min}$$
Use sample SD (`ddof=1`) and the original samples for raw RMS.
The supplied identity explains the connection; no derivation is required:
$$x_{\mathrm{RMS}}^2=\bar{x}^2+\frac{N-1}{N}s^2.$$

**Completed example -> your calculation:** the next cell calculates X-channel
features from the same cut. Keep it unchanged. In your task cell, use the same
four expressions with `vibration_y` instead of `reference_x`.
`ddof=1` requests sample SD; `**2` squares each value, and `np.sqrt` takes a square root.
All X and Y comparison values here are in **g**, so compare the same feature across channels.

In [ ]:
# [GUIDED - RUN UNCHANGED]
# reference_x contains the X acceleration samples (g) for a same-cut comparison.
reference_x = data['vibration_x_g'].to_numpy()
x_mean = np.mean(reference_x)  # Average signed acceleration, in g.
x_sd = np.std(reference_x, ddof=1)  # Fluctuation around the X mean, in g.
x_rms = np.sqrt(np.mean(reference_x**2))  # Overall X magnitude relative to zero, in g.
x_p2p = np.max(reference_x) - np.min(reference_x)  # Observed X range, in g.
print('X reference mean / sample SD / RMS / peak-to-peak (g):')
print(x_mean, x_sd, x_rms, x_p2p)

In [ ]:
# Apply the same operations to vibration_y; all answers are in g.
mean_value = None  # TODO: Find the average signed vibration level.
sd_value = None  # TODO: Find sample SD around the mean.
rms_value = None  # TODO: Find raw RMS relative to zero.
peak_to_peak = None  # TODO: Find maximum minus minimum.

In [ ]:
# [GUIDED - RUN UNCHANGED]
print('Mean (g):', mean_value)
print('Sample SD (g):', sd_value)
print('Raw RMS (g):', rms_value)
print('Peak-to-peak (g):', peak_to_peak)

## Part 3 response (1–2 sentences)

Compare your Y-channel SD and peak-to-peak with the supplied X reference. Which channel has the larger values in this cut? Cite both pairs of values (g) and keep the claim limited to this recording.

**TODO:** Write your response here.

# Part 4 — FFT and STFT Interpretation — 20 points

**Time guide: 20 minutes.**

![Whole-record FFT and windowed STFT](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab03/fft_vs_stft.png)

Conceptual comparison; compare actual Vibration Y signals from c1 cuts 1 and 315 below. All processing and plotting code is supplied.

**Fair comparison:** both records use the same sensor, sampling rate and
processing settings. FFTs share frequency and magnitude limits; STFTs share
frequency limits and one color scale. Each record keeps its own physical duration.
No plot is divided by its own maximum. These are observations of two records;
cut number alone cannot establish that wear caused every difference.

In [ ]:
# [GUIDED - RUN UNCHANGED]
# cut1_data supplies the early reference only; Parts 1–3 continue to use cut 315.
CUT1_URL = 'https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/data/phm2010/c1_selected_cuts/c1_cut001.csv'
cut1_data = pd.read_csv(CUT1_URL, float_precision='round_trip')
y_cut1 = cut1_data['vibration_y_g'].to_numpy()  # Early-cut acceleration in g.
cut1_duration_s = len(y_cut1) / fs  # Nominal physical duration, in seconds.
print('Cut 1 / cut 315 nominal durations (s):', cut1_duration_s, len(data)/fs)

In [ ]:
# [GUIDED - RUN UNCHANGED]
# Center each record separately, then use the same periodic Hann window rule.
centered1 = y_cut1 - np.mean(y_cut1)  # Remove the early record's average, in g.
window1 = hann(len(centered1), sym=False)
spectrum1 = rfft(centered1 * window1)
frequency1_hz = rfftfreq(len(centered1), d=1/fs)  # Early record's own frequency grid.
# One-sided magnitude in g: divide by window sum, not the record's largest peak.
magnitude1_g = np.abs(spectrum1) / np.sum(window1)
magnitude1_g[1:1+(len(centered1)-1)//2] *= 2  # Exclude DC and Nyquist if present.

In [ ]:
# [GUIDED - RUN UNCHANGED]
if vibration_y is None:
    print('Complete the earlier required inputs, then rerun this cell.')
else:
    # The longer record has its own FFT grid but the same magnitude scaling.
    centered = vibration_y - np.mean(vibration_y)
    window = hann(len(centered), sym=False)
    spectrum = rfft(centered * window)
    frequency_hz = rfftfreq(len(centered), d=1/fs)
    amplitude_g = np.abs(spectrum) / np.sum(window)  # One-sided FFT magnitude (g).
    amplitude_g[1:1+(len(centered)-1)//2] *= 2

**FFT magnitude:** the supplied scaling accounts for record length and Hann
window gain, so magnitudes are in g rather than unscaled FFT coefficient sizes.
This preserves the difference between records; neither curve is peak-normalized.
Different record lengths still give different frequency resolution. Compare broad
components and approximate heights, not exact bin-to-bin matches. A peak does not
by itself prove a mechanical mode or a wear mechanism.

In [ ]:
# [GUIDED - RUN UNCHANGED]
if vibration_y is None:
    print('Complete the earlier required inputs, then rerun this cell.')
else:
    # Shared axes let peak heights be compared directly in physical units.
    fft_top_g = 1.1 * max(magnitude1_g.max(), amplitude_g.max())
    fig, (ax_fft1, ax_fft315) = plt.subplots(2, 1, figsize=(9,6), sharex=True, sharey=True)
    ax_fft1.plot(frequency1_hz/1000, magnitude1_g, linewidth=0.7)
    ax_fft315.plot(frequency_hz/1000, amplitude_g, linewidth=0.7)
    ax_fft1.set(title='Cut 1: Vibration Y FFT', ylabel='Magnitude (g)')
    ax_fft315.set(title='Cut 315: Vibration Y FFT', xlabel='Frequency (kHz)',
                  ylabel='Magnitude (g)', xlim=(0,25), ylim=(0,fft_top_g))
    fig.tight_layout()
    plt.show()

In [ ]:
# [GUIDED - RUN UNCHANGED]
nperseg = 2048  # Samples analyzed in each short window.
noverlap = 1536  # Samples shared by neighboring windows.
hop = 512  # Window-start advance: 2048 - 1536 samples.
# Early record: frequency (Hz), window-center time (s), power density (g²/Hz).
stft1_hz, frame1_s, psd1 = spectrogram(
    centered1, fs=fs, window='hann', nperseg=nperseg, noverlap=noverlap,
    nfft=nperseg, detrend=False, scaling='density', mode='psd')
db1 = 10*np.log10(np.maximum(psd1, 1e-20))  # Log density, with a numerical floor.

In [ ]:
# [GUIDED - RUN UNCHANGED]
if vibration_y is None:
    print('Complete the earlier required inputs, then rerun this cell.')
else:
    # Late record: identical analysis settings, no per-window detrending or padding.
    stft_hz, frame_s, psd = spectrogram(
        centered, fs=fs, window='hann', nperseg=nperseg, noverlap=noverlap,
        nfft=nperseg, detrend=False, scaling='density', mode='psd')
    db = 10*np.log10(np.maximum(psd, 1e-20))
    # One common display range for BOTH records; do not rescale each image separately.
    color_max_db = max(db1.max(), db.max())
    color_min_db = color_max_db - 80
    comparison_end_s = max(cut1_duration_s, len(data)/fs)

In [ ]:
# [GUIDED - RUN UNCHANGED]
if vibration_y is None:
    print('Complete the earlier required inputs, then rerun this cell.')
else:
    # Shared time axis keeps the shorter recording visibly shorter; blank means no data.
    fig, (ax_stft1, ax_stft315) = plt.subplots(2, 1, figsize=(9,6),
                                            sharex=True, sharey=True, layout='constrained')
    image1 = ax_stft1.pcolormesh(frame1_s, stft1_hz/1000, db1, shading='auto',
        vmin=color_min_db, vmax=color_max_db, cmap='viridis', rasterized=True)
    image315 = ax_stft315.pcolormesh(frame_s, stft_hz/1000, db, shading='auto',
        vmin=color_min_db, vmax=color_max_db, cmap='viridis', rasterized=True)
    ax_stft1.set(title='Cut 1: Vibration Y STFT', ylabel='Frequency (kHz)')
    ax_stft315.set(title='Cut 315: Vibration Y STFT', xlabel='Window-center time (s)',
                   ylabel='Frequency (kHz)', ylim=(0,25), xlim=(0,comparison_end_s))
    fig.colorbar(image315, ax=[ax_stft1,ax_stft315], label='PSD (dB re 1 g²/Hz)')
    plt.show()

**How to compare the figures**
1. Read one component visible in both FFTs and compare its approximate magnitude
   in g. Also look for a region that is more prominent in one record. Both vertical
   axes have exactly the same limits; a taller peak has greater plotted magnitude.
2. Locate those frequency regions in both STFTs. The **same color means the same
   PSD value** in both panels. Explain one observed band-strength or time-pattern
   difference, using approximate frequencies and local times for each recording.
3. Cut 1 ends earlier. Its blank right-hand area is **unrecorded time, not silence
   or zero vibration**. Times begin at each record's own start; they are not aligned
   cutting phases. Hann windows describe short intervals, not single instants.
4. Separate observation from cause. Two records cannot isolate wear from other
   possible changes, and these figures do not establish predictive performance.

Write 3–4 sentences: one common frequency component, a magnitude comparison,
a supported STFT comparison, and a limitation. Combine related points as needed.
Example structure: "Both show a component near __ kHz. Its magnitude is roughly
__ g in cut 1 versus __ g in cut 315. The common-color STFT shows __ at __.
This comparison alone cannot establish __." No new code or derivation is required.

## Part 4 response (3–4 sentences)

Compare cut 1 and cut 315 using a common frequency component, FFT magnitudes in g, and one STFT band/time observation. State a limit on the conclusions that two records support.

**TODO:** Write your response here.

# Part 5 — Feature Relationship and Simple Selection — 20 points

**Time guide: 20 minutes.**

## 5A. One correlation you calculate
We now change scale: one row of `master` represents **one complete cut**, not a
sample. `wear_mean_um` is mean flute wear in micrometers, the continuous target.
The table contains 945 labeled cuts from three cutters. Features summarize full
records; do not correlate the raw time-sample rows with this cut-level target.

Pearson **r** describes linear association and has no unit. Its sign indicates
direction; its absolute value indicates linear strength. For example, -0.8 is a
stronger linear association than +0.2. A value near zero can still hide nonlinear
or cutter-dependent information. Correlation is not causation or tested prediction.

The scatter uses cutter colors so you can see that pooling different cutters can
hide distinct patterns. All correlations below are pooled exploratory summaries;
no training, test split or model performance is being evaluated.

In [ ]:
# [GUIDED - RUN UNCHANGED]
# master holds cut-level features and wear; cutter_id is source metadata.
MASTER_URL = 'https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/data/phm2010/features/phm2010_features.csv'
master = pd.read_csv(MASTER_URL, float_precision='round_trip')
print('Master shape (cuts, columns):', master.shape)
# Supplied filters retain one cutter per color, without student-written grouping.
c1 = master[master['cutter_id']=='c1']
c4 = master[master['cutter_id']=='c4']
c6 = master[master['cutter_id']=='c6']

In [ ]:
# [GUIDED - RUN UNCHANGED]
# Each point links one cut's mean wear (µm) to its Y acceleration SD (g).
fig, ax = plt.subplots(figsize=(9,4.5))
ax.scatter(c1['wear_mean_um'], c1['vibration_y_sd'], s=8, label='c1')
ax.scatter(c4['wear_mean_um'], c4['vibration_y_sd'], s=8, label='c4')
ax.scatter(c6['wear_mean_um'], c6['vibration_y_sd'], s=8, label='c6')
ax.set(title='Vibration Y SD versus wear', xlabel='Mean wear (µm)', ylabel='Vibration Y SD (g)')
ax.legend()
fig.tight_layout()
plt.show()

**One-line pattern:** `table["feature"].corr(table["target"], method="pearson")`.
For example, `master["force_y_mean"].corr(master["wear_mean_um"], method="pearson")`
would calculate the association for Force Y mean. Adapt this expression to
**Vibration Y SD**. Do not change the target or calculate 28 values by hand.

In [ ]:
# target_r is the unitless pooled association between Y vibration SD and wear.
target_r = None  # TODO: Adapt the supplied expression to vibration_y_sd and wear_mean_um.
print('Pearson r:', target_r)

## 5B. Explore all seven channels using supplied code
**GUIDED - RUN UNCHANGED.** Each channel contributes the same four familiar
features: mean, sample SD, raw RMS, and peak-to-peak. There are **7 × 4 = 28**
candidates. Extra crest-factor/kurtosis columns and metadata are excluded.

The seven channels are Force X/Y/Z (N), Vibration X/Y/Z (g), and the processed
AE-RMS channel (V). For AE-RMS, "RMS" in a column such as `ae_rms_rms` means a
summary calculated across the already processed AE-RMS signal, not a raw AE waveform.

Read the heatmap **one row at a time**: compare the four summaries for that sensor.
Then compare rows. Each cell is a feature's correlation **with wear**, not a
correlation between two features. The color scale retains negative and positive
values. Your value from 5A should match row Vibration Y, column Sample SD.

Column-name guide: combine the row prefix (`force_x`, `force_y`, `force_z`,
`vibration_x`, `vibration_y`, `vibration_z`, `ae_rms`) with the feature ending
(`_mean`, `_sd`, `_rms`, `_peak_to_peak`). The complete allowed names are supplied
in the lists below; copy names from them when selecting features.

In [ ]:
# [GUIDED - RUN UNCHANGED]
# Explicit lists identify the four existing columns for each force channel.
force_x_features = ['force_x_mean', 'force_x_sd', 'force_x_rms', 'force_x_peak_to_peak']
force_y_features = ['force_y_mean', 'force_y_sd', 'force_y_rms', 'force_y_peak_to_peak']
force_z_features = ['force_z_mean', 'force_z_sd', 'force_z_rms', 'force_z_peak_to_peak']

In [ ]:
# [GUIDED - RUN UNCHANGED]
# Each list keeps mean, SD, RMS, peak-to-peak in exactly that order.
vibration_x_features = ['vibration_x_mean', 'vibration_x_sd',
                        'vibration_x_rms', 'vibration_x_peak_to_peak']
vibration_y_features = ['vibration_y_mean', 'vibration_y_sd',
                        'vibration_y_rms', 'vibration_y_peak_to_peak']
vibration_z_features = ['vibration_z_mean', 'vibration_z_sd',
                        'vibration_z_rms', 'vibration_z_peak_to_peak']
ae_features = ['ae_rms_mean', 'ae_rms_sd', 'ae_rms_rms', 'ae_rms_peak_to_peak']

In [ ]:
# [GUIDED - RUN UNCHANGED]
# Adding lists joins their names; it does not add the sensor measurements.
candidate_features = (force_x_features + force_y_features + force_z_features
                      + vibration_x_features + vibration_y_features
                      + vibration_z_features + ae_features)
# corrwith applies the same Pearson calculation to each of the 28 columns.
wear_r = master[candidate_features].corrwith(master['wear_mean_um'], method='pearson')
# Reshape only the displayed correlations: seven sensor rows, four feature columns.
channel_labels = ['Force X', 'Force Y', 'Force Z', 'Vibration X',
                  'Vibration Y', 'Vibration Z', 'AE-RMS']
feature_labels = ['Mean', 'Sample SD', 'RMS', 'Peak-to-peak']
wear_map = pd.DataFrame(wear_r.to_numpy().reshape(7,4),
                        index=channel_labels, columns=feature_labels)
display(wear_map.round(3))

In [ ]:
# [GUIDED - RUN UNCHANGED]
# The supplied loops add one readable number to each feature-versus-wear cell.
fig, ax = plt.subplots(figsize=(9,6))
image = ax.imshow(wear_map, vmin=-1, vmax=1, cmap='coolwarm', aspect='auto')
ax.set_xticks(range(4), feature_labels)
ax.set_yticks(range(7), channel_labels)
ax.set_title('All-channel feature correlations with mean wear')
for row in range(7):
    for column in range(4):
        ax.text(column, row, f'{wear_map.iloc[row,column]:.2f}',
                ha='center', va='center', fontsize=12)
fig.colorbar(image, ax=ax, label='Pearson r with wear (dimensionless)')
fig.tight_layout()
plt.show()

## 5C. Choose three and check their overlap
1. Pick **three distinct column names** from the 28 candidates. Look for meaningful
   wear association, but do not simply choose the three largest absolute values.
2. Put the quoted names in a list separated by commas, using the same format as
   the supplied lists. Only this list is your coding task; it contains three items.
3. Run the following supplied **3 × 3** heatmap. Here each cell compares **two selected
   features**. Ignore the diagonal (each feature compared with itself is 1).
   Off-diagonal values close to +1 or -1 suggest overlapping linear information.
4. If two choices overlap strongly, consider replacing one and rerun this cell and
   the heatmap. You may retain an overlapping pair if you give a specific reason.
   There is no fixed cutoff and no single correct set. Submit only your final list.

For example, SD and RMS of the same near-zero-mean vibration channel may carry
almost the same information. Different channels can also overlap; inspect your
actual matrix. A lower-|r| feature is allowed if you explain its physical role.
No 28 × 28 matrix, feature ranking, extra response, or model is required.

In [ ]:
# Copy three distinct full column names from the provided candidate lists.
selected_features = None  # TODO: Replace None with a list of three quoted candidate names.
print('Selected features:', selected_features)

In [ ]:
# [GUIDED - RUN UNCHANGED]
if selected_features is None:
    print('Complete the earlier required inputs, then rerun this cell.')
else:
    # selected_corr describes overlap within your final three choices, not with wear.
    selected_corr = master[selected_features].corr(method='pearson')
    fig, ax = plt.subplots(figsize=(9,5))
    image = ax.imshow(selected_corr, vmin=-1, vmax=1, cmap='coolwarm')
    ax.set_xticks(range(3), selected_features, rotation=20, ha='right')
    ax.set_yticks(range(3), selected_features)
    ax.set_title('Overlap among your three selected features')
    for row in range(3):
        for column in range(3):
            ax.text(column, row, f'{selected_corr.iloc[row,column]:.2f}',
                    ha='center', va='center', fontsize=12)
    fig.colorbar(image, ax=ax, label='Feature-feature Pearson r')
    fig.tight_layout()
    plt.show()

**Response guide:** in sentence 1, use the 7 × 4 heatmap to cite wear-correlation
values and name the physical information your choices represent. In sentence 2,
use at least one off-diagonal value from your 3 × 3 matrix to discuss overlap.
Use a third sentence, if helpful, for a tradeoff or limitation. Rounded values
are sufficient. The same reasoning can support different defensible selections;
a large pooled correlation does not guarantee generalization to another cutter.

## Part 5 response (2–3 sentences)

Justify your final three features using wear association, physical meaning, and at least one selected-pair correlation. Keep the claim exploratory rather than claiming an optimal predictor set.

**TODO:** Write your response here.

# Submission checklist
Reserve approximately 10 minutes. Rendered Markdown checkboxes are not clickable;
edit `[ ]` to `[x]` to record completion.

- [ ] Name and WSU AccessID are entered.
- [ ] All 10 code entries and five short responses are complete.
- [ ] All six required figures are visible: waveform, FFT, STFT, wear scatter, 7 × 4 wear heatmap, and 3 × 3 selected-feature heatmap.
- [ ] Numerical values and figures have the required labels/units.
- [ ] Restart and Run all succeeds without unfinished-input reminders.
- [ ] Exported PDF has readable, unclipped code, equations and figures.
- [ ] Submit HW02_Firstname_Lastname.ipynb and HW02_Firstname_Lastname.pdf in Canvas.

Keep required code and outputs visible. No separate report. Canvas shows the
official due date and submission policies; no deadline is specified here.

## References
- [PHM Society 2010 challenge](https://phmsociety.org/phm_competition/2010-phm-society-conference-data-challenge/)
- [Course data provenance and rights](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/README.md)
- [Feature definitions](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/DATA_DICTIONARY.md)
- [SciPy spectrogram](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.spectrogram.html)